<a href="https://colab.research.google.com/github/petrovortex/foundations_of_ml_course/blob/main/hometask_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from pandas import DataFrame

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

seed = 42

In [ ]:
df_train = pd.read_csv('train_processed.csv')
X_test = pd.read_csv('test_processed.csv')

In [ ]:
y = df_train['target']
X = df_train.drop(labels=['target'], axis=1)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.8, random_state=seed, stratify=y)

In [ ]:
X_val, _, y_val, _ = train_test_split(X_val, y_val, test_size=0.8, random_state=seed, stratify=y_val)

## 1. Разведочный анализ и подготовка данных

In [ ]:
X_train.head()

,id,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,...,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37
5966,6885,1701.400933,1642.0,435870,435883.0,377,0.023,56,35659,26.742904,...,95.353431,84.469422,0.419832,0.402288,0.010416,0.000494,Very_High,Very_Low,Cat_035,type_b_special
8895,6437,1167.430657,1248.0,2412964,2412966.0,64,0.018,11,7446,110.748984,...,136.160314,133.917458,0.791031,0.802213,0.020527,0.037942,Very_High,Medium,Cat_037,type_b_special
10964,3108,1410.498212,1395.0,1672180,1672222.0,576,NaN,35,60260,118.656520,...,141.644096,143.025262,0.346957,0.292995,NaN,NaN,Medium,Medium,Cat_008,TYPE_C
5622,7153,474.609729,531.0,1211728,NaN,527,0.056,37,63525,96.665412,...,141.710844,137.239683,0.263553,0.316610,0.054043,0.097266,Very_High,Medium,Cat_003,type_e
3081,7427,11.194398,90.0,3928519,3928612.0,3256,0.070,43,334869,94.516507,...,127.359885,121.899973,0.564658,0.545340,0.061690,0.110643,Medium,Medium,Cat_027,Type_A


In [ ]:
X_val.info()

<class 'pandas.core.frame.DataFrame'>
Index: 430 entries, 448 to 713
Data columns (total 38 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          430 non-null    int64  
 1   feature_01  430 non-null    float64
 2   feature_02  348 non-null    float64
 3   feature_03  430 non-null    int64  
 4   feature_04  365 non-null    float64
 5   feature_05  430 non-null    int64  
 6   feature_06  367 non-null    float64
 7   feature_07  430 non-null    int64  
 8   feature_08  430 non-null    int64  
 9   feature_09  430 non-null    float64
 10  feature_10  430 non-null    float64
 11  feature_11  430 non-null    int64  
 12  feature_12  430 non-null    int64  
 13  feature_13  430 non-null    int64  
 14  feature_14  360 non-null    float64
 15  feature_15  430 non-null    float64
 16  feature_16  430 non-null    float64
 17  feature_17  430 non-null    float64
 18  feature_18  430 non-null    float64
 19  feature_19  430 non-null    floa

In [ ]:
df_info = DataFrame({
    'nunique': X_train.nunique(),
    'type': X_train.dtypes,
    'nulls': len(X_train) - X_train.count()
})

In [ ]:
df_info

,nunique,type,nulls
id,10762,int64,0
feature_01,10762,float64,0
feature_02,1131,float64,1565
feature_03,2647,int64,0
feature_04,2477,float64,1616
feature_05,1033,int64,0
feature_06,404,float64,1615
feature_07,309,int64,0
feature_08,2191,int64,0
feature_09,10762,float64,0


1. Заполним пропуски. Пропуски есть только в числовых признаках, заменим наны на медиану.
2. Некоторые числовые признаки принимают небольшое число значений. Выберем такие (например, у которых меньше 100 уникальных значений) и сделаем из них строки (для катбустинга).
3. Удалим id и попробуем определить другие неинформативные признаки

In [ ]:
def preprocess_data(data, encoder, scaler, for_cb=False, is_test=False):

    ids = data['id']
    data = data.drop(columns=['id'], axis=1)

    numeric_columns = data.select_dtypes(include=['int64', 'float64']).columns
    categorical_columns = data.select_dtypes(include=['object']).columns

    data[numeric_columns] = data[numeric_columns].fillna(data[numeric_columns].median())

    cat_features = []

    if for_cb:
        for column in data.columns:
            if data[column].nunique() < 100 and data[column].dtype in ['float64', 'int64']:
                data[column] = data[column].apply(str)

        for column in data.columns:
            if data[column].dtype == 'object':
                cat_features.append(column)
                data[column] = data[column].apply(str)
    else:
        if is_test:
            encoded_data = encoder.transform(data[categorical_columns])
        else:
            encoded_data = encoder.fit_transform(data[categorical_columns])

        encoded_columns = encoder.get_feature_names_out(categorical_columns)
        encoded_df = pd.DataFrame(encoded_data, columns=encoded_columns, index=data.index)
        data = pd.concat([data[numeric_columns], encoded_df], axis=1)

        if is_test:
            data = pd.DataFrame(
                scaler.transform(data),
                columns=data.columns,
                index=data.index
            )
        else:
            data = pd.DataFrame(
                scaler.fit_transform(data),
                columns=data.columns,
                index=data.index
            )


    return data, ids, cat_features, encoder, scaler

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
scaler = StandardScaler()

In [ ]:
X_train_cb, train_ids, cat_features, encoder, scaler = preprocess_data(X_train, encoder, scaler, for_cb=True)
X_val_cb, val_ids, _, encoder, scaler = preprocess_data(X_val, encoder, scaler, for_cb=True, is_test=True)

In [ ]:
X_train_prep, train_ids, _, encoder, scaler = preprocess_data(X_train, encoder, scaler)
X_val_prep, val_ids, _, encoder, scaler = preprocess_data(X_val, encoder, scaler, is_test=True)

In [ ]:
X_train.head()

,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,...,feature_36_Cat_047,feature_36_Cat_048,feature_36_Cat_049,feature_37_Type A+,feature_37_Type_A,feature_37_Type_D,feature_37_type_b,feature_37_type_b_special,feature_37_type_e,feature_37_unknown
9058,-0.900984,-1.595581,-0.091824,-0.061426,-0.406597,-0.412432,-0.505953,-0.428393,-0.977778,-2.444016,...,-0.142437,-0.144461,-0.141415,-0.376317,-0.388608,-0.380009,-0.367576,2.611747,-0.375673,-0.377121
1309,-0.128250,0.077818,0.226337,0.284466,-0.375158,-0.272648,-0.386318,-0.379972,0.859191,-0.072121,...,-0.142437,-0.144461,-0.141415,-0.376317,-0.388608,-0.380009,-0.367576,-0.382886,2.661888,-0.377121
10621,0.142019,0.162575,-0.214166,-0.219947,-0.411965,-0.435729,-0.495984,-0.421540,-0.085595,-0.212273,...,-0.142437,-0.144461,-0.141415,2.657334,-0.388608,-0.380009,-0.367576,-0.382886,-0.375673,-0.377121
249,-0.703288,-1.317406,-0.493402,-0.498013,0.197906,0.531109,1.158971,0.236740,1.083029,0.429887,...,-0.142437,-0.144461,-0.141415,-0.376317,-0.388608,-0.380009,-0.367576,2.611747,-0.375673,-0.377121
6174,0.457645,0.712406,-0.830605,-0.864619,-0.396629,-0.342540,-0.376348,-0.412433,-0.126128,-1.198842,...,-0.142437,-0.144461,-0.141415,-0.376317,-0.388608,-0.380009,2.720524,-0.382886,-0.375673,-0.377121


## 2. Обучение и тестирование модели

### 2.1 Catboost

In [ ]:
! pip install catboost -q

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
catboost_classifier = CatBoostClassifier(random_state=seed, verbose=False)
catboost_classifier.fit(X_train_cb, y_train, cat_features=cat_features)

In [ ]:
y_pred = catboost_classifier.predict(X_val_cb)

In [ ]:
accuracy_score(y_val, y_pred)

0.8094648166501487

### 2.2 TabPFN

In [ ]:
!pip install tabpfn

In [ ]:
y_pred = tabpfn_classifier.predict(X_val_cb)

In [ ]:
accuracy_score(y_val, y_pred)

0.8134291377601586

### 2.3 Тестирование и сабмит

In [ ]:
X_test_cb, test_ids, _, encoder, scaler = preprocess_data(X_test, encoder, scaler, is_test=True, for_cb=True)

In [ ]:
y_test = tabpfn_classifier.predict(X_test_cb)

In [ ]:
results_df = pd.DataFrame({'id': test_ids, 'target': y_test})
results_df.to_csv(f'submission_seed_{seed}.csv', index=False)